# Paper Figure 1 – model comparison

The YAML file controls subjects, models, checkpoints, simulation size and output path.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.paper_fig1 import run_evaluation, results_dataframe

CONFIG = PROJECT_ROOT / 'configs/PaperFig1/model_comparison.yaml'

In [ ]:
import numpy as np

brain_mask = np.load("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/Denoising/datasets/Proton/7T/B0corrected_wo_LipidMask/Vol1_UCSF/masks/brain_mask.npy")
lipid_mask = np.load("/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/Denoising/datasets/Proton/7T/B0corrected_wo_LipidMask/Vol1_UCSF/masks/lipid_mask.npy")

In [ ]:
import numpy as np
from scipy.ndimage import binary_dilation, generate_binary_structure


def dilate_lipid_inward(brain_mask, lipid_mask, n_voxels):
    """
    Expand lipid mask inward into the brain mask by n voxel layers.

    Parameters
    ----------
    brain_mask : ndarray, bool
        Brain mask.
    lipid_mask : ndarray, bool
        Lipid/scalp mask.
    n_voxels : int
        Number of brain voxel layers transferred to lipid mask.

    Returns
    -------
    brain_out : ndarray, bool
        Reduced brain mask.
    lipid_out : ndarray, bool
        Expanded lipid mask.
    transferred : ndarray, bool
        Brain voxels that were reassigned to lipid.
    """

    brain = np.asarray(brain_mask, dtype=bool).copy()
    lipid = np.asarray(lipid_mask, dtype=bool).copy()

    if brain.shape != lipid.shape:
        raise ValueError("brain_mask and lipid_mask must have same shape")

    if np.any(brain & lipid):
        raise ValueError("brain_mask and lipid_mask must initially be disjoint")

    # 6-neighbour connectivity in 3D:
    # only voxels sharing a face count as neighbours
    structure = generate_binary_structure(3, 1)

    original_brain = brain.copy()

    for _ in range(n_voxels):
        new_layer = binary_dilation(
            lipid,
            structure=structure
        ) & brain

        if not np.any(new_layer):
            break

        lipid[new_layer] = True
        brain[new_layer] = False

    transferred = original_brain & lipid

    return brain, lipid, transferred

import matplotlib.pyplot as plt

def plot_masks_before_after(
    brain_before,
    lipid_before,
    brain_after,
    lipid_after,
    z
):
    fig, ax = plt.subplots(2, 2, figsize=(8, 8))

    ax[0, 0].imshow(brain_before[:, :, z].T, origin="lower")
    ax[0, 0].set_title("Brain mask – before")

    ax[0, 1].imshow(lipid_before[:, :, z].T, origin="lower")
    ax[0, 1].set_title("Lipid mask – before")

    ax[1, 0].imshow(brain_after[:, :, z].T, origin="lower")
    ax[1, 0].set_title("Brain mask – after")

    ax[1, 1].imshow(lipid_after[:, :, z].T, origin="lower")
    ax[1, 1].set_title("Lipid mask – after")

    for a in ax.flat:
        a.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
n = 5
brain_degraded, lipid_degraded, transferred = dilate_lipid_inward(brain_mask, lipid_mask, n)

plot_masks_before_after(
    brain_mask,
    lipid_mask,
    brain_degraded,
    lipid_degraded,
    z=15
)

In [ ]:
from pathlib import Path
from time import perf_counter

from walinet.training_data.lipid_removal import compute_lipid_projection_operator

data_path = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/"
    "Denoising/datasets/Proton/7T/B0corrected_wo_LipidMask/"
    "Vol1_UCSF/OriginalData/data.npy"
)
n_timepoints = 840
gpu_index = 0  # Select the CUDA device used for the benchmark.
csi_fids = np.load(data_path, mmap_mode="r")
lipid_selection = np.asarray(lipid_mask, dtype=bool)
lipid_fids = np.asarray(csi_fids[lipid_selection, :n_timepoints], dtype=np.complex64)
lipid_spectra = np.fft.fftshift(
    np.fft.fft(lipid_fids, axis=-1), axes=-1
).astype(np.complex64, copy=False)
spectra_for_operator = lipid_spectra[:, None, None, :]
operator_mask = np.ones(spectra_for_operator.shape[:-1], dtype=bool)

operators = {}
timings = {}
for method in ("legacy", "eigenvalue", "gpu"):
    start = perf_counter()
    operators[method] = compute_lipid_projection_operator(
        spectra_for_operator, operator_mask, method=method, gpu_index=gpu_index
    )
    timings[method] = perf_counter() - start

difference = operators["eigenvalue"] - operators["legacy"]
print(f"Legacy:     {timings['legacy']:.3f} s")
print(f"Eigenvalue: {timings['eigenvalue']:.3f} s")
print(f"CPU eigenvalue speed-up: {timings['legacy'] / timings['eigenvalue']:.2f}x")
print(f"GPU speed-up:            {timings['legacy'] / timings['gpu']:.2f}x")
print(f"Maximum absolute difference: {np.max(np.abs(difference)):.3e}")
print(f"Relative Frobenius difference: {np.linalg.norm(difference) / np.linalg.norm(operators['legacy']):.3e}")
print("CPU eigenvalue equivalent:", np.allclose(
    operators["eigenvalue"], operators["legacy"], rtol=1e-5, atol=1e-6
))
gpu_difference = operators["gpu"] - operators["legacy"]
print(f"GPU maximum absolute difference: {np.max(np.abs(gpu_difference)):.3e}")
print(f"GPU relative Frobenius difference: {np.linalg.norm(gpu_difference) / np.linalg.norm(operators['legacy']):.3e}")
print("GPU equivalent:", np.allclose(
    operators["gpu"], operators["legacy"], rtol=1e-4, atol=1e-5
))


In [ ]:
# Timing of the default operator call (GPU with automatic CPU fallback).
from pathlib import Path
from time import perf_counter

from walinet.training_data.lipid_removal import compute_lipid_projection_operator

gpu_index = 0
n_timepoints = 840
gpu_data_path = Path(
    "/ceph/mri.meduniwien.ac.at/departments/radiology/mrsbrain/public/hfish/"
    "Denoising/datasets/Proton/7T/B0corrected_wo_LipidMask/"
    "Vol1_UCSF/OriginalData/data.npy"
)
gpu_csi_fids = np.load(gpu_data_path, mmap_mode="r")
gpu_lipid_fids = np.asarray(
    gpu_csi_fids[np.asarray(lipid_mask, dtype=bool), :n_timepoints],
    dtype=np.complex64,
)
gpu_lipid_spectra = np.fft.fftshift(
    np.fft.fft(gpu_lipid_fids, axis=-1), axes=-1
).astype(np.complex64, copy=False)
gpu_spectra_for_operator = gpu_lipid_spectra[:, None, None, :]
gpu_operator_mask = np.ones(
    gpu_spectra_for_operator.shape[:-1], dtype=bool
)

start = perf_counter()
default_operator = compute_lipid_projection_operator(
    gpu_spectra_for_operator,
    gpu_operator_mask,
    gpu_index=gpu_index,
)
default_time = perf_counter() - start
print(f"Default operator call: {default_time:.3f} s")
